# PaddleOCR-VL 1.6 — CC-OCR Vietnamese trên Kaggle

Benchmark zero-shot hệ OCR đầu cuối. Notebook này tải đúng checkpoint chính thức và ghi từng ảnh để có thể resume. `SMOKE_TEST` gồm 5 ảnh đầu theo thứ tự TSV; `FULL_BENCHMARK` gồm mọi ảnh hợp lệ. Không suy diễn kết quả trước khi chạy GPU thực tế.

## 1. Kiểm tra môi trường trước cài đặt

Đọc Python, Torch, CUDA, driver, CC, VRAM, RAM, đĩa và internet. Dừng nếu `cuda:0` không phải T4. Hai T4 có VRAM riêng; notebook chỉ dùng GPU 0.

In [ ]:
from pathlib import Path
import sys
LIB = Path('/kaggle/working/ocr_benchmark_lib')
LIB.mkdir(parents=True, exist_ok=True)
(LIB / 'benchmark_core.py').write_text('"""Shared, fail-closed dataset and measurement code for Kaggle OCR notebooks."""\nfrom __future__ import annotations\n\nimport base64\nimport csv\nimport gc\nimport hashlib\nimport io\nimport json\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport threading\nimport time\nimport unicodedata\nimport urllib.request\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Any, Callable\n\nDATASET_ID = "wulipc/CC-OCR"\nDATASET_FILE = "multi_lan_ocr/Vietnamese/Vietnamese_150.tsv"\nEXPECTED_COLUMNS = {"index", "image", "question", "answer"}\nMODEL_IDS = {"paddle": "PaddlePaddle/PaddleOCR-VL-1.6", "deepseek": "deepseek-ai/DeepSeek-OCR-2"}\nTERMINAL = {"completed", "failed", "oom", "timeout"}\n\n\ndef write_json(path: Path, data: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding="utf-8")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as fh:\n        for block in iter(lambda: fh.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef environment() -> dict[str, Any]:\n    import psutil\n    import torch\n    result: dict[str, Any] = {\n        "python": platform.python_version(), "platform": platform.platform(),\n        "torch": torch.__version__, "torch_cuda_runtime": torch.version.cuda,\n        "cuda_available": torch.cuda.is_available(),\n        "ram_total_bytes": psutil.virtual_memory().total,\n        "ram_available_bytes": psutil.virtual_memory().available,\n        "disk_free_bytes": shutil.disk_usage("/kaggle/working").free,\n        "gpu": [], "internet": False,\n    }\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            p = torch.cuda.get_device_properties(i)\n            free, total = torch.cuda.mem_get_info(i)\n            result["gpu"].append({"index": i, "name": p.name, "compute_capability": list(p.major_minor) if hasattr(p, "major_minor") else [p.major, p.minor], "total_vram_bytes": total, "free_vram_bytes": free})\n    try:\n        result["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,memory.used,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10).stdout.strip()\n    except (OSError, subprocess.TimeoutExpired) as exc:\n        result["nvidia_smi_error"] = str(exc)\n    try:\n        with urllib.request.urlopen("https://huggingface.co", timeout=10) as response:\n            result["internet"] = response.status == 200\n    except Exception as exc:\n        result["internet_error"] = str(exc)\n    return result\n\n\ndef require_t4(env: dict[str, Any]) -> None:\n    if not env["cuda_available"] or not env["gpu"]:\n        raise RuntimeError("GPU CUDA không khả dụng; dừng benchmark GPU.")\n    gpu = env["gpu"][0]\n    if "T4" not in gpu["name"] or gpu["compute_capability"] != [7, 5]:\n        raise RuntimeError(f"Cần NVIDIA T4 trên cuda:0, thực tế: {gpu}")\n\n\ndef download_tsv(workdir: Path, revision: str | None = None) -> tuple[Path, str]:\n    from huggingface_hub import HfApi, hf_hub_download\n    revision = revision or HfApi().dataset_info(DATASET_ID).sha\n    path = Path(hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename=DATASET_FILE, revision=revision, local_dir=workdir / "hf_dataset"))\n    return path, revision\n\n\ndef inspect_tsv(path: Path) -> dict[str, Any]:\n    """Read actual schema before decoding. Never print Base64 image data."""\n    csv.field_size_limit(2**31 - 1)  # Base64 image field can be megabytes.\n    with path.open("r", encoding="utf-8-sig", newline="") as fh:\n        reader = csv.DictReader(fh, delimiter="\\t")\n        fields = reader.fieldnames or []\n        if not EXPECTED_COLUMNS.issubset(fields):\n            raise ValueError(f"TSV schema mismatch: {fields}; required {sorted(EXPECTED_COLUMNS)}")\n        preview = []\n        for row in reader:\n            preview.append({"index": row.get("index"), "question": row.get("question"), "answer": row.get("answer"), "image_chars": len(row.get("image") or "")})\n            if len(preview) == 5:\n                break\n    return {"columns": fields, "preview": preview, "tsv_sha256": sha256(path)}\n\n\ndef make_manifest(path: Path, outdir: Path) -> tuple[list[dict[str, str]], list[dict[str, str]]]:\n    from PIL import Image, UnidentifiedImageError\n    csv.field_size_limit(2**31 - 1)\n    images = outdir / "images"\n    images.mkdir(parents=True, exist_ok=True)\n    rows: list[dict[str, str]] = []\n    rejected: list[dict[str, str]] = []\n    seen_ids: set[str] = set()\n    seen_hashes: dict[str, str] = {}\n    with path.open("r", encoding="utf-8-sig", newline="") as fh:\n        reader = csv.DictReader(fh, delimiter="\\t")\n        if not EXPECTED_COLUMNS.issubset(reader.fieldnames or []):\n            raise ValueError("TSV schema changed; inspect before parsing")\n        for ordinal, item in enumerate(reader):\n            sample_id = str(item.get("index", "")).strip()\n            reason = ""\n            try:\n                if not sample_id or sample_id in seen_ids:\n                    raise ValueError("empty or duplicate sample ID")\n                seen_ids.add(sample_id)\n                answer = item.get("answer")\n                if not isinstance(answer, str) or not answer.strip():\n                    raise ValueError("empty ground truth")\n                encoded = item.get("image")\n                if not isinstance(encoded, str) or not encoded.strip():\n                    raise ValueError("empty image")\n                if encoded.startswith("data:"):\n                    encoded = encoded.split(",", 1)[1]\n                payload = base64.b64decode(encoded, validate=True)\n                with Image.open(io.BytesIO(payload)) as img:\n                    img.verify()\n                with Image.open(io.BytesIO(payload)) as img:\n                    width, height = img.size\n                    fmt = (img.format or "PNG").lower()\n                    if width < 1 or height < 1:\n                        raise ValueError("invalid dimensions")\n                digest = hashlib.sha256(payload).hexdigest()\n                suffix = ".jpg" if fmt in {"jpeg", "jpg"} else f".{fmt}"\n                image_path = images / f"{ordinal:03d}{suffix}"\n                image_path.write_bytes(payload)\n                rows.append({"sample_id": sample_id, "ordinal": str(ordinal), "image_path": str(image_path), "image_sha256": digest,\n                             "width": str(width), "height": str(height), "ground_truth": answer, "question": item.get("question", ""),\n                             "duplicate_of": seen_hashes.get(digest, "")})\n                seen_hashes.setdefault(digest, sample_id)\n            except (ValueError, TypeError, KeyError, UnidentifiedImageError, OSError, base64.binascii.Error) as exc:\n                reason = str(exc)\n            if reason:\n                rejected.append({"ordinal": str(ordinal), "sample_id": sample_id, "reason": reason})\n    if not rows:\n        raise RuntimeError("No valid samples")\n    fields = list(rows[0])\n    with (outdir / "manifest.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fields)\n        writer.writeheader(); writer.writerows(rows)\n    with (outdir / "rejected_samples.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=["ordinal", "sample_id", "reason"])\n        writer.writeheader(); writer.writerows(rejected)\n    return rows, rejected\n\n\ndef manifest_fingerprint(rows: list[dict[str, str]]) -> str:\n    canonical = [{k: row[k] for k in ("sample_id", "image_sha256", "width", "height", "ground_truth")}\n                 for row in rows]\n    payload = json.dumps(canonical, ensure_ascii=False, sort_keys=True).encode("utf-8")\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef normalize(text: str, mode: str) -> str:\n    if mode == "strict":\n        return text\n    if mode != "normalized":\n        raise ValueError(mode)\n    return " ".join(unicodedata.normalize("NFC", text).replace("\\r\\n", "\\n").replace("\\r", "\\n").split())\n\n\ndef plain_text(raw: str) -> str:\n    """Conservative Markdown removal; keep visible characters and punctuation."""\n    raw = re.sub(r"!\\[[^]]*\\]\\([^)]*\\)", "", raw)\n    raw = re.sub(r"\\[([^]]+)\\]\\([^)]*\\)", r"\\1", raw)\n    raw = re.sub(r"(?m)^\\s{0,3}(?:#{1,6}\\s+|>\\s+|[-*+]\\s+|\\d+\\.\\s+)", "", raw)\n    raw = raw.replace("**", "").replace("__", "").replace("`", "")\n    return raw\n\n\ndef _distance(a: list[str] | str, b: list[str] | str) -> int:\n    try:\n        from rapidfuzz.distance import Levenshtein\n        return Levenshtein.distance(a, b)\n    except ImportError:\n        previous = list(range(len(b) + 1))\n        for i, left in enumerate(a, start=1):\n            current = [i]\n            for j, right in enumerate(b, start=1):\n                current.append(min(current[-1] + 1, previous[j] + 1, previous[j-1] + (left != right)))\n            previous = current\n        return previous[-1]\n\n\ndef score(records: list[dict[str, Any]], mode: str) -> dict[str, Any]:\n    if not records:\n        return {"status": "NOT_EVALUATED", "n": 0}\n    chars_ref = chars_edit = words_ref = words_edit = exact = empty = failed = 0\n    per_image = []\n    for row in records:\n        gt = normalize(row["ground_truth"], mode)\n        ok = row["status"] == "completed"\n        pred = normalize(row.get("prediction_text", "") if ok else "", mode)\n        failed += not ok\n        empty += not pred\n        exact += pred == gt and ok\n        cd = _distance(gt, pred)\n        wd = _distance(gt.split(), pred.split())\n        chars_ref += len(gt); chars_edit += cd\n        words_ref += len(gt.split()); words_edit += wd\n        per_image.append(cd / max(1, len(gt)))\n    n = len(records)\n    return {"status": "MEASURED", "n": n, "corpus_cer": chars_edit / max(1, chars_ref),\n            "corpus_whitespace_wer": words_edit / max(1, words_ref), "median_per_image_cer": median(per_image),\n            "exact_match_rate": exact / n, "empty_prediction_rate": empty / n, "runtime_error_rate": failed / n,\n            "failed_policy": "failed/oom/timeout => empty prediction for end-to-end quality; reported separately"}\n\n\ndef read_jsonl(path: Path) -> list[dict[str, Any]]:\n    if not path.exists():\n        return []\n    rows = []\n    with path.open(encoding="utf-8") as fh:\n        for line in fh:\n            if line.strip():\n                rows.append(json.loads(line))\n    return rows\n\n\ndef append_jsonl(path: Path, row: dict[str, Any]) -> None:\n    with path.open("a", encoding="utf-8") as fh:\n        fh.write(json.dumps(row, ensure_ascii=False, default=str) + "\\n")\n        fh.flush(); os.fsync(fh.fileno())\n\n\nclass DeviceVRAMSampler:\n    """Sample whole-device NVML usage; includes other processes and may miss short spikes."""\n    def __init__(self) -> None:\n        self.peak: int | None = None\n        self.stop = threading.Event()\n        self.thread: threading.Thread | None = None\n        try:\n            import pynvml\n            pynvml.nvmlInit()\n            self.pynvml = pynvml\n            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)\n        except Exception:\n            self.pynvml = None\n\n    def start(self) -> None:\n        if self.pynvml is None:\n            return\n        def poll() -> None:\n            while not self.stop.is_set():\n                try:\n                    used = int(self.pynvml.nvmlDeviceGetMemoryInfo(self.handle).used)\n                    self.peak = max(self.peak or 0, used)\n                except Exception:\n                    pass\n                self.stop.wait(.05)\n        self.thread = threading.Thread(target=poll, daemon=True)\n        self.thread.start()\n\n    def finish(self) -> int | None:\n        self.stop.set()\n        if self.thread:\n            self.thread.join(timeout=1)\n        return self.peak\n\n\ndef export_predictions(rows: list[dict[str, Any]], outdir: Path) -> None:\n    fields = ["sample_id", "image_sha256", "ground_truth", "status", "prediction_raw", "prediction_text", "latency_s", "peak_allocated_bytes", "peak_reserved_bytes", "peak_device_vram_bytes", "ram_rss_bytes", "truncated", "error"]\n    with (outdir / "predictions.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader(); writer.writerows(rows)\n    with (outdir / "failures.csv").open("w", encoding="utf-8", newline="") as fh:\n        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader(); writer.writerows(r for r in rows if r["status"] != "completed")\n\n\ndef performance(rows: list[dict[str, Any]]) -> dict[str, Any]:\n    import numpy as np\n    good = [float(r["latency_s"]) for r in rows if r["status"] == "completed"]\n    all_times = [float(r["latency_s"]) for r in rows]\n    peaks = [int(r["peak_device_vram_bytes"]) for r in rows if r.get("peak_device_vram_bytes") is not None]\n    return {"completed_count": len(good), "total_inference_seconds": sum(all_times),\n            "throughput_images_per_second": len(good) / sum(all_times) if all_times else None,\n            "latency_p50_s": float(np.percentile(good, 50)) if good else None,\n            "latency_p90_s": float(np.percentile(good, 90)) if good else None,\n            "latency_p95_s": float(np.percentile(good, 95)) if good else None,\n            "peak_device_vram_bytes": max(peaks) if peaks else None,\n            "vram_note": "NVML whole-device sampled at 50 ms; includes other processes and can miss brief peaks"}\n\n\ndef run_benchmark(manifest: list[dict[str, str]], outdir: Path, infer: Callable[[Path], tuple[str, bool | None]],\n                  sync: Callable[[], None], gpu_peak: Callable[[], tuple[int | None, int | None, int | None]],\n                  mode: str, warmup: Path, max_seconds: float = 300.0,\n                  reset_peak: Callable[[], None] | None = None) -> list[dict[str, Any]]:\n    """Append a terminal record after each sample. Warm-up is excluded from measured rows."""\n    import psutil\n    outdir.mkdir(parents=True, exist_ok=True)\n    chosen = manifest[:5] if mode == "SMOKE_TEST" else manifest\n    path = outdir / "predictions.jsonl"\n    old = read_jsonl(path)\n    ids = [(x["sample_id"], x["image_sha256"]) for x in chosen]\n    if len(ids) != len(set(ids)):\n        raise RuntimeError("duplicate sample keys")\n    allowed = set(ids)\n    if any((r["sample_id"], r["image_sha256"]) not in allowed for r in old):\n        raise RuntimeError("checkpoint contains different dataset/mode")\n    if len({(r["sample_id"], r["image_sha256"]) for r in old}) != len(old):\n        raise RuntimeError("duplicate checkpoint rows; resolve manually")\n    if any(r.get("status") not in TERMINAL for r in old):\n        raise RuntimeError("checkpoint contains unknown status")\n    done = {(r["sample_id"], r["image_sha256"]) for r in old if r["status"] in TERMINAL}\n    if len(done) < len(chosen):\n        sync(); start = time.perf_counter()\n        infer(warmup); sync()\n        warmup_result = {"image_path": str(warmup), "seconds": time.perf_counter() - start,\n                         "resume": bool(old)}\n        write_json(outdir / "warmup.json", warmup_result)\n        append_jsonl(outdir / "warmup_history.jsonl", warmup_result)\n    for item in chosen:\n        key = (item["sample_id"], item["image_sha256"])\n        if key in done:\n            continue\n        sync()\n        if reset_peak:\n            reset_peak()\n        sampler = DeviceVRAMSampler()\n        sampler.start()\n        start = time.perf_counter()\n        record: dict[str, Any] = {"sample_id": key[0], "image_sha256": key[1], "ground_truth": item["ground_truth"], "status": "failed", "prediction_raw": "", "prediction_text": "", "truncated": False, "error": ""}\n        try:\n            raw, truncated = infer(Path(item["image_path"]))\n            if not isinstance(raw, str):\n                raise TypeError(f"inference returned {type(raw).__name__}; expected str")\n            sync()\n            record.update(status="completed", prediction_raw=raw, prediction_text=plain_text(raw), truncated=truncated)\n        except Exception as exc:\n            sync()\n            record.update(status="oom" if "out of memory" in str(exc).lower() else "failed", error=f"{type(exc).__name__}: {exc}"[:2000])\n            gc.collect()\n            try:\n                import torch\n                torch.cuda.empty_cache()\n            except ImportError:\n                pass\n        elapsed = time.perf_counter() - start\n        device_peak = sampler.finish()\n        if elapsed > max_seconds and record["status"] == "completed":\n            record["status"] = "timeout"  # post hoc threshold; no hard preemption\n        a, r, d = gpu_peak()\n        record.update(latency_s=elapsed, peak_allocated_bytes=a, peak_reserved_bytes=r, peak_device_vram_bytes=device_peak or d,\n                      ram_rss_bytes=psutil.Process().memory_info().rss)\n        append_jsonl(path, record)\n        done.add(key)\n        if record["status"] == "oom":\n            break  # repeated OOM is unsafe; preserve checkpoint\n    rows = read_jsonl(path)\n    export_predictions(rows, outdir)\n    write_json(outdir / "metrics.json", {"run_status": "MEASURED" if len(rows) == len(chosen) else "PARTIAL",\n                                         "expected_samples": len(chosen), "recorded_samples": len(rows),\n                                         **{m: score(rows, m) for m in ("strict", "normalized")},\n                                         "performance": performance(rows)})\n    return rows\n', encoding='utf-8')
(LIB / 'model_adapters.py').write_text('"""Official-checkpoint adapters. DeepSeek T4 patch is explicit and revision-bound."""\nfrom __future__ import annotations\n\nimport hashlib\nimport inspect\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nPADDLE_ID = "PaddlePaddle/PaddleOCR-VL-1.6"\nDEEPSEEK_ID = "deepseek-ai/DeepSeek-OCR-2"\nDEEPSEEK_PROMPT = "<image>\\nFree OCR. "\nMAX_NEW_TOKENS = 2048\n\n\ndef install_common() -> None:\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",\n                           "huggingface_hub==0.36.0", "Pillow==11.3.0", "psutil==7.0.0", "rapidfuzz==3.14.1", "pandas==2.2.3", "matplotlib==3.10.3", "nvidia-ml-py==12.575.51"])\n\n\ndef install_paddle(env: dict[str, Any]) -> None:\n    """Match an official CUDA wheel to current runtime; stop for unknown combinations."""\n    cuda = env.get("torch_cuda_runtime") or ""\n    major = cuda.split(".")[0]\n    if major == "12":\n        index = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"\n        if int(cuda.split(".")[1]) < 6:\n            raise RuntimeError("CUDA <12.6: choose an official Paddle wheel manually; notebook will not guess")\n    else:\n        raise RuntimeError(f"No audited Paddle wheel for Kaggle torch CUDA {cuda}; inspect official install matrix")\n    subprocess.check_call([sys.executable, "-m", "pip", "install", "paddlepaddle-gpu==3.2.1", "-i", index])\n    subprocess.check_call([sys.executable, "-m", "pip", "install", \'paddleocr[doc-parser]==3.7.0\'])\n\n\nclass PaddleAdapter:\n    def __init__(self) -> None:\n        from paddleocr import PaddleOCRVL\n        self.init_start = time.perf_counter()\n        self.pipeline = PaddleOCRVL(pipeline_version="v1.6", device="gpu:0", use_layout_detection=True,\n                                    use_doc_orientation_classify=False, use_doc_unwarping=False,\n                                    use_queues=False, markdown_ignore_labels=[])\n        self.init_seconds = time.perf_counter() - self.init_start\n\n    def infer(self, image: Path) -> tuple[str, None]:\n        results = list(self.pipeline.predict(str(image), max_new_tokens=MAX_NEW_TOKENS))\n        if len(results) != 1:\n            raise RuntimeError(f"Expected one page result; got {len(results)}")\n        md = results[0].markdown\n        if not isinstance(md, dict) or "markdown_texts" not in md:\n            raise TypeError(f"Unexpected Paddle markdown structure: {type(md).__name__}")\n        texts = md["markdown_texts"]\n        if isinstance(texts, str):\n            return texts, None  # Pipeline does not expose a reliable truncation flag.\n        if isinstance(texts, list) and all(isinstance(x, str) for x in texts):\n            return "\\n".join(texts), None\n        raise TypeError(f"Unexpected markdown_texts: {type(texts).__name__}")\n\n    @staticmethod\n    def sync() -> None:\n        import paddle\n        paddle.device.cuda.synchronize()\n\n    @staticmethod\n    def gpu_peak() -> tuple[int | None, int | None, None]:\n        import paddle\n        api = paddle.device.cuda\n        try:\n            return int(api.max_memory_allocated(0)), int(api.max_memory_reserved(0)), None\n        except (AttributeError, RuntimeError, TypeError):\n            return None, None, None\n\n    @staticmethod\n    def reset_peak() -> None:\n        import paddle\n        try:\n            paddle.device.cuda.reset_peak_memory_stats(0)\n        except (AttributeError, RuntimeError, TypeError):\n            pass\n\n\ndef patch_deepseek_for_t4(snapshot: Path, max_new_tokens: int = MAX_NEW_TOKENS) -> dict[str, Any]:\n    """Patch only known hard-coded inference choices, with exact-count guards."""\n    target = snapshot / "modeling_deepseekocr2.py"\n    source = target.read_text(encoding="utf-8")\n    patch_log = snapshot / "t4_patch.json"\n    if patch_log.exists():\n        previous = json.loads(patch_log.read_text(encoding="utf-8"))\n        if hashlib.sha256(source.encode()).hexdigest() == previous.get("patched_sha256"):\n            return previous\n    original_hash = hashlib.sha256(source.encode()).hexdigest()\n    # Five executable BF16 sites plus one comment in the reviewed upstream file.\n    if source.count("torch.bfloat16") != 6 or source.count("max_new_tokens=8192") != 2:\n        raise RuntimeError("DeepSeek official source changed; audit T4 patch before continuing")\n    source = source.replace("torch.bfloat16", "torch.float16")\n    source = source.replace("max_new_tokens=8192", f"max_new_tokens={max_new_tokens}")\n    target.write_text(source, encoding="utf-8")\n    result = {"file": target.name, "original_sha256": original_hash,\n            "patched_sha256": hashlib.sha256(source.encode()).hexdigest(),\n            "changes": ["BF16 tensors and autocast -> FP16", f"max_new_tokens 8192 -> {max_new_tokens}"],\n            "risk": "FP16 is an unvalidated precision port; output quality may differ from official BF16 baseline"}\n    patch_log.write_text(json.dumps(result, indent=2), encoding="utf-8")\n    return result\n\n\nclass DeepSeekAdapter:\n    def __init__(self, cache_dir: Path, revision: str | None = None) -> None:\n        import torch\n        from huggingface_hub import HfApi, snapshot_download\n        from transformers import AutoModel, AutoTokenizer\n        if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):\n            raise RuntimeError("DeepSeek T4 path requires CC 7.5")\n        self.revision = revision or HfApi().model_info(DEEPSEEK_ID).sha\n        self.snapshot = Path(snapshot_download(repo_id=DEEPSEEK_ID, revision=self.revision, local_dir=cache_dir / "deepseek_model"))\n        self.patch = patch_deepseek_for_t4(self.snapshot)\n        self.init_start = time.perf_counter()\n        self.tokenizer = AutoTokenizer.from_pretrained(str(self.snapshot), trust_remote_code=True, local_files_only=True)\n        # torch_dtype applies while loading; avoid a transient FP32 GPU copy.\n        self.model = AutoModel.from_pretrained(str(self.snapshot), trust_remote_code=True, local_files_only=True,\n                                               use_safetensors=True, torch_dtype=torch.float16,\n                                               _attn_implementation="eager", low_cpu_mem_usage=True).eval().to("cuda:0")\n        if self.model.config._attn_implementation != "eager":\n            raise RuntimeError("eager attention was not selected")\n        self.init_seconds = time.perf_counter() - self.init_start\n\n    def infer(self, image: Path) -> tuple[str, bool]:\n        # eval_mode returns decoded text according to the official source; save_results=False avoids eval() branches.\n        text = self.model.infer(self.tokenizer, prompt=DEEPSEEK_PROMPT, image_file=str(image),\n                                output_path="/kaggle/working/deepseek_tmp", base_size=1024,\n                                image_size=768, crop_mode=True, save_results=False, eval_mode=True)\n        if not isinstance(text, str):\n            raise TypeError(f"model.infer returned {type(text).__name__}")\n        # Exact termination is unavailable through infer(); report a conservative flag.\n        truncated = len(self.tokenizer.encode(text, add_special_tokens=False)) >= MAX_NEW_TOKENS\n        return text, truncated\n\n    @staticmethod\n    def sync() -> None:\n        import torch\n        torch.cuda.synchronize(0)\n\n    @staticmethod\n    def gpu_peak() -> tuple[int, int, None]:\n        import torch\n        return torch.cuda.max_memory_allocated(0), torch.cuda.max_memory_reserved(0), None\n\n    @staticmethod\n    def reset_peak() -> None:\n        import torch\n        torch.cuda.reset_peak_memory_stats(0)\n', encoding='utf-8')
sys.path.insert(0, str(LIB))
print('Embedded benchmark source ready:', sorted(p.name for p in LIB.iterdir()))

In [ ]:
from pathlib import Path
from benchmark_core import environment, require_t4, write_json
ROOT = Path('/kaggle/working/vietnamese_ocr')
ROOT.mkdir(parents=True, exist_ok=True)
ENV = environment()
write_json(ROOT / 'environment_preinstall.json', ENV)
print(ENV)
require_t4(ENV)
if not ENV['internet']: raise RuntimeError('Kaggle Internet chưa bật')

## 2. Dependencies được pin và kiểm tra lại

Giữ Torch hiện có của Kaggle; không nâng cấp toàn bộ môi trường. Paddle dùng wheel CUDA 12.6 đã tài liệu hóa. DeepSeek dùng Transformers 4.46.3 theo model card. Nếu CUDA hoặc ABI khác, dừng và điều tra thay vì tự đổi phiên bản.

In [ ]:
from model_adapters import install_common, install_paddle
install_common()
install_paddle(ENV)
import importlib.metadata as im
print({p: im.version(p) for p in ['torch','huggingface_hub','rapidfuzz','pandas']})

## 3. Dataset: schema, ảnh và nhãn

Tải TSV bằng `hf_hub_download` theo commit SHA. Đầu tiên in tên cột và bản xem trước không chứa Base64. Sau đó giải mã, xác thực ảnh, ghi `manifest.csv` và mọi mẫu lỗi. `answer` chỉ được dùng làm ground truth sau khi người chạy xem ảnh và xác nhận ý nghĩa của cột; không đưa nó vào prompt.

In [ ]:
from benchmark_core import download_tsv, inspect_tsv, make_manifest, manifest_fingerprint, sha256
from IPython.display import display
from PIL import Image
import pandas as pd
DATASET_REVISION_OVERRIDE = None  # Set to the other notebook's commit SHA when reproducing
TSV, DATASET_REVISION = download_tsv(ROOT, DATASET_REVISION_OVERRIDE)
SCHEMA = inspect_tsv(TSV)
write_json(ROOT / 'dataset_schema.json', SCHEMA)
print('Columns:', SCHEMA['columns'], 'Dataset revision:', DATASET_REVISION)
display(pd.DataFrame(SCHEMA['preview']))
MANIFEST, REJECTED = make_manifest(TSV, ROOT / 'dataset')
print('Valid:', len(MANIFEST), 'Rejected:', len(REJECTED), 'Duplicate images:', sum(bool(x['duplicate_of']) for x in MANIFEST))
for row in MANIFEST[:5]:
    print('ID:', row['sample_id'], 'Question:', row['question'], 'Answer:', row['ground_truth'])
    display(Image.open(row['image_path']))
print('Inspect the five images above before confirming answer semantics.')

In [ ]:
CONFIRM_GROUND_TRUTH_SEMANTICS = False  # Set True only after visual and question/answer review
assert CONFIRM_GROUND_TRUTH_SEMANTICS, 'Confirm that answer is the literal image transcription'
MODE = 'SMOKE_TEST'  # After inspecting smoke outputs, change to FULL_BENCHMARK and rerun
assert MODE in {'SMOKE_TEST','FULL_BENCHMARK'}
import random, numpy as np, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
OUT = ROOT / ('paddle' if True else 'deepseek') / MODE.lower()
OUT.mkdir(parents=True, exist_ok=True)
import shutil
shutil.copy2(ROOT / 'dataset' / 'manifest.csv', OUT / 'manifest.csv')
shutil.copy2(ROOT / 'dataset' / 'rejected_samples.csv', OUT / 'rejected_samples.csv')
shutil.copytree(ROOT / 'dataset' / 'images', OUT / 'images', dirs_exist_ok=True)

## 4. Nạp checkpoint chính thức và kiểm chứng adapter

Paddle chạy pipeline `PaddleOCRVL(pipeline_version='v1.6')` có layout analysis. DeepSeek dùng checkpoint `deepseek-ai/DeepSeek-OCR-2`, eager attention và FP16 trên T4. Bản vá DeepSeek được hash và ghi rõ trong `run_manifest.json`; FP16 có thể làm thay đổi độ chính xác so với BF16 chính thức.

In [ ]:
from model_adapters import PaddleAdapter, PADDLE_ID, MAX_NEW_TOKENS
from huggingface_hub import HfApi
MODEL_REVISION = HfApi().model_info(PADDLE_ID).sha
ADAPTER = PaddleAdapter()
MODEL_ID = PADDLE_ID
PATCH = None
from benchmark_core import manifest_fingerprint
RUN_META = {'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'dataset_id': 'wulipc/CC-OCR', 'dataset_file': 'multi_lan_ocr/Vietnamese/Vietnamese_150.tsv', 'dataset_revision': DATASET_REVISION, 'dataset_sha256': SCHEMA['tsv_sha256'], 'manifest_sha256': manifest_fingerprint(MANIFEST), 'valid_sample_count': len(MANIFEST), 'rejected_sample_count': len(REJECTED), 'mode': MODE, 'batch_size': 1, 'seed': 42, 'max_new_tokens': MAX_NEW_TOKENS, 'device': 'cuda:0', 'initialization_seconds': ADAPTER.init_seconds, 'patch': PATCH, 'precision': 'Paddle default' if True else 'FP16', 'preprocessing': 'official PaddleOCRVL pipeline' if True else 'DeepSeek official dynamic crop 1024/768', 'normalization': ['strict','normalized NFC whitespace'], 'ground_truth_confirmed': CONFIRM_GROUND_TRUTH_SEMANTICS}
previous_path = OUT / 'run_manifest.json'
if previous_path.exists():
    import json
    previous = json.loads(previous_path.read_text(encoding='utf-8'))
    for key in ['model_id','model_revision','dataset_revision','dataset_sha256','manifest_sha256','mode','precision','patch','max_new_tokens']:
        if previous[key] != RUN_META[key]: raise RuntimeError(f'Resume config mismatch: {key}')
else:
    write_json(previous_path, RUN_META)
write_json(OUT / 'benchmark_config.json', RUN_META)
write_json(OUT / 'environment.json', environment())
print(RUN_META)

## 5. Warm-up, smoke test và benchmark

Warm-up dùng ảnh đầu tiên, không tính trong latency chính. Mỗi ảnh đồng bộ CUDA trước/sau đo, ghi checkpoint JSONL ngay sau inference. `timeout` là ngưỡng hậu kiểm, không cắt ngang lệnh GPU đang treo. OOM dừng sau một mẫu để tránh lặp lỗi. Kết quả lỗi được giữ lại và tính như chuỗi rỗng trong CER/WER đầu cuối.

In [ ]:
from benchmark_core import run_benchmark
ROWS = run_benchmark(MANIFEST, OUT, ADAPTER.infer, ADAPTER.sync, ADAPTER.gpu_peak, MODE, Path(MANIFEST[0]['image_path']), reset_peak=ADAPTER.reset_peak)
print('Recorded:', len(ROWS), '/', 5 if MODE == 'SMOKE_TEST' else len(MANIFEST))
display(pd.DataFrame(ROWS)[['sample_id','status','latency_s','error']].head(10))

## 6. Xem ảnh và xuất kết quả

Kiểm tra output thô cùng ảnh và nhãn. `metrics.json` chứa cả strict và normalized; đây là kết quả của chế độ hiện tại, không ghép smoke với full. Lưu version của notebook để tải toàn bộ `/kaggle/working/vietnamese_ocr` từ Kaggle Notebook Outputs.

In [ ]:
from benchmark_core import score
for row in ROWS[:5]:
    match = next(x for x in MANIFEST if x['sample_id'] == row['sample_id'])
    display(Image.open(match['image_path']))
    print('GT:', row['ground_truth'])
    print('Prediction:', row['prediction_raw'])
    print('Status:', row['status'], 'Error:', row['error'])
print('Strict:', score(ROWS, 'strict'))
print('Normalized:', score(ROWS, 'normalized'))
print('Outputs:', sorted(p.name for p in OUT.iterdir()))